In [1]:
#Code written by Guillermo Martínez-Ara and Marina Marchenko following K. Isihara indications

import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from skimage import io, measure
from skimage.measure import label, regionprops
from scipy import ndimage as ndi
from scipy.stats import mode
import trimesh

In [2]:
print(trimesh.__version__)

4.11.2


In [3]:
#mesh quantification
def measure_meshes(entire_mesh):
    meshes = entire_mesh.split()
    Vs = [np.abs(mesh.volume) for mesh in meshes]
    As = [mesh.area for mesh in meshes]
    Ms = [mesh.integral_mean_curvature for mesh in meshes]
    Xs = [mesh.euler_number for mesh in meshes]
    df = pd.DataFrame(data={'volume': Vs, 'area': As, 'IMC': Ms, 'euler char': Xs})
    df['R0_lumen'] = np.sqrt(df['area']/4/np.pi)
    df['reducedVolume'] = df['volume']/(4/3*np.pi)/np.power(df['R0_lumen'],3)
    df['reducedCurvature'] = df['IMC']/(4*np.pi*df['R0_lumen'])
    df['sphericity'] = np.power(np.pi,1/3)*np.power(6*df['volume'],2/3)/df['area']
    return df, meshes

def spherocylinder():
    """returns reduced volume and reduced area difference of spherocylinder family"""
    gamma = np.linspace(0.001,350,1000)
    v_spheretube = np.divide(3/4*gamma+1, np.power(1+gamma/2, 1.5))
    m_spheretube = np.divide(gamma+4, np.sqrt(1+gamma/2))*np.pi
    return v_spheretube, m_spheretube
v_sc, m_sc = spherocylinder()

#Input the resolution of isometric image
zcali, ycali, xcali = (4, 4, 4)
spacing = [zcali, ycali, xcali]

In [9]:
#1. select folder stim or control
bigdir = r''
outputdir = r''
bigdirlist = os.listdir(bigdir)

#file = ".DS_Store"
#path = os.path.join(bigdir, file)
#os.remove(path)

#bigdir = r'\\ebisuya.embl.es\ebisuya\Marusja\Results\Viventis\230124_D8-D11_opt3_400cells_control\Segmented'
#bigdirlist = os.listdir(bigdir)
numfolders = len(bigdirlist)

for t in range(numfolders):
    print(t)
    lumenlist = []
    loadfile = os.path.join(bigdir, bigdirlist[t])
            
    img = io.imread(loadfile)
        
            #We know segmented lumens are in channel 2, segmented organoid in channel 3:
            
    mask = img[:,:,:]
        
    label_img = label(mask, connectivity=mask.ndim)
    numcols = np.max(label_img)
            #randomcol = rand_cmap(numcols+10, type = 'soft')
        
            #Let's make the mesh
    verts, faces, ___, ___ = measure.marching_cubes(mask, spacing = spacing, gradient_direction = 'ascent')
    entire_mesh = trimesh.Trimesh(vertices=verts, faces=faces)
    entire_mesh.faces.shape
        
            #mesh quantification:
            
    df, meshes = measure_meshes(entire_mesh)
        
            #We have decided to eliminate all lumens below 30000 fl. (30nl)
            
        #filtered = df[df.volume > 30000]
        
            #let's plot all lumens: 
            
    lumenlist.append(df)
            #medianlist.append(filtered.median())
        #numberoflumens.append(len(filtered))
            
    lumendf = pd.concat(lumenlist, axis = 1)
        #meandf = pd.concat(meanlist, axis = 1)
        

    exportname = os.path.join(outputdir, bigdirlist[t])
            
    lumendf.to_csv(exportname + "_lumens.csv")

0
1
2
3
